# 01 - Data Exploration

Exploring the Construction Site Safety dataset before training.
- Split statistics
- Class distribution
- Sample images with annotations
- Bounding box size analysis


In [9]:
import os
import random
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns

sns.set_theme(style="whitegrid")

# Detect environment
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    DATASET_DIR = Path('/content/css-data')
else:
    DATASET_DIR = Path(r"D:\Interview\SecondRoundOriginalgit\construction-safety-monitor\data\dataset\css-data")

CLASS_NAMES = {
    0: "Hardhat",       1: "Mask",          2: "NO-Hardhat",
    3: "NO-Mask",       4: "NO-Safety Vest", 5: "Person",
    6: "Safety Cone",   7: "Safety Vest",   8: "machinery",
    9: "vehicle"
}
CLASS_COLORS = {
    0: "#00CC00", 1: "#00CC00", 2: "#FF0000",
    3: "#FF0000", 4: "#FF0000", 5: "#FF8C00",
    6: "#FFFF00", 7: "#00CC00", 8: "#888888", 9: "#888888"
}

print(f"Running in Colab : {IN_COLAB}")
print(f"Dataset path     : {DATASET_DIR}")
print(f"Exists           : {DATASET_DIR.exists()}")


Running in Colab : True
Dataset path     : /content/css-data
Exists           : False


In [13]:
if IN_COLAB:
    import zipfile
    from google.colab import drive
    drive.mount('/content/drive')

    ZIP_PATH = Path('/content/drive/MyDrive/Task/archive.zip')
    
    if not DATASET_DIR.exists():
        print("Unzipping archive.zip ...")
        with zipfile.ZipFile(ZIP_PATH, 'r') as z:
            z.extractall('/content/')
        print("Done!")
    
    print(f"Dataset exists: {DATASET_DIR.exists()}")
else:
    print("Running locally - no unzip needed")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Unzipping archive.zip ...
Done!
Dataset exists: True


## 1. Split Statistics

How many images in each split?

In [14]:
for split in ["train", "valid", "test"]:
    img_dir = DATASET_DIR / split / "images"
    lbl_dir = DATASET_DIR / split / "labels"
    if img_dir.exists():
        n_img = len(list(img_dir.glob("*")))
        n_lbl = len(list(lbl_dir.glob("*.txt")))
        print(f"{split:>6}: {n_img:>5} images | {n_lbl:>5} label files")


 train:  2605 images |  2605 label files
 valid:   114 images |   114 label files
  test:    82 images |    82 label files


## 2. Class Distribution

In [ ]:
# Count class instances across all splits
class_counts = Counter()
split_class_counts = {}
bbox_sizes = []  # (class_id, width, height)

for split in ['train', 'val', 'test']:
    lbl_dir = DATASET_DIR / split / 'labels'
    if not lbl_dir.exists():
        continue
    split_counter = Counter()
    for label_file in lbl_dir.glob('*.txt'):
        for line in label_file.read_text().strip().split('\n'):
            if not line.strip():
                continue
            parts = line.strip().split()
            cls_id = int(parts[0])
            class_counts[cls_id] += 1
            split_counter[cls_id] += 1
            if len(parts) == 5:
                w, h = float(parts[3]), float(parts[4])
                bbox_sizes.append((cls_id, w, h))
    split_class_counts[split] = split_counter

# Display as bar chart
fig, ax = plt.subplots(figsize=(10, 5))
classes = sorted(class_counts.keys())
counts = [class_counts[c] for c in classes]
names = [f'{c}: {CLASS_NAMES.get(c, "unknown")}' for c in classes]
colors = [CLASS_COLORS.get(c, '#999999') for c in classes]

bars = ax.bar(names, counts, color=colors, edgecolor='black', alpha=0.8)
ax.set_ylabel('Number of Instances')
ax.set_title('Class Distribution (All Splits)')

for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            str(count), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Class distribution per split
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, split in zip(axes, ['train', 'val', 'test']):
    if split not in split_class_counts:
        continue
    counter = split_class_counts[split]
    c_ids = sorted(counter.keys())
    c_vals = [counter[c] for c in c_ids]
    c_names = [CLASS_NAMES.get(c, str(c)) for c in c_ids]
    c_colors = [CLASS_COLORS.get(c, '#999') for c in c_ids]
    ax.bar(c_names, c_vals, color=c_colors, edgecolor='black', alpha=0.8)
    ax.set_title(f'{split} split')
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Class Distribution per Split', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Bounding Box Size Analysis

In [ ]:
# Analyze bbox sizes (normalized widths and heights)
if bbox_sizes:
    bbox_arr = np.array(bbox_sizes)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Width distribution
    for cls_id in sorted(set(bbox_arr[:, 0].astype(int))):
        mask = bbox_arr[:, 0] == cls_id
        axes[0].hist(bbox_arr[mask, 1], bins=50, alpha=0.5,
                     label=CLASS_NAMES.get(int(cls_id), str(int(cls_id))))
    axes[0].set_xlabel('Normalized Width')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Bounding Box Width Distribution')
    axes[0].legend()
    
    # Height distribution
    for cls_id in sorted(set(bbox_arr[:, 0].astype(int))):
        mask = bbox_arr[:, 0] == cls_id
        axes[1].hist(bbox_arr[mask, 2], bins=50, alpha=0.5,
                     label=CLASS_NAMES.get(int(cls_id), str(int(cls_id))))
    axes[1].set_xlabel('Normalized Height')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Bounding Box Height Distribution')
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()
    
    # Size categories (COCO-style)
    areas = bbox_arr[:, 1] * bbox_arr[:, 2]
    small = (areas < 0.01).sum()
    medium = ((areas >= 0.01) & (areas < 0.05)).sum()
    large = (areas >= 0.05).sum()
    print(f'Small objects (area < 1%):  {small} ({small/len(areas)*100:.1f}%)')
    print(f'Medium objects (1-5%):      {medium} ({medium/len(areas)*100:.1f}%)')
    print(f'Large objects (area > 5%):  {large} ({large/len(areas)*100:.1f}%)')

## 4. Sample Images with Annotations

In [ ]:
def show_annotated_image(img_path, lbl_path, ax):
    """Display an image with its YOLO annotations."""
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    
    ax.imshow(img)
    
    if lbl_path.exists():
        for line in lbl_path.read_text().strip().split('\n'):
            if not line.strip():
                continue
            parts = line.strip().split()
            cls_id = int(parts[0])
            cx, cy, bw, bh = [float(p) for p in parts[1:5]]
            
            # Convert normalized to pixel coordinates
            x1 = (cx - bw/2) * w
            y1 = (cy - bh/2) * h
            box_w = bw * w
            box_h = bh * h
            
            color = CLASS_COLORS.get(cls_id, '#FFFFFF')
            rect = patches.Rectangle((x1, y1), box_w, box_h,
                                      linewidth=2, edgecolor=color,
                                      facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, y1 - 5, CLASS_NAMES.get(cls_id, str(cls_id)),
                    color=color, fontsize=8, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.7))
    
    ax.set_title(img_path.name, fontsize=8)
    ax.axis('off')


# Show sample images from train set
train_imgs = list((DATASET_DIR / 'train' / 'images').glob('*'))
if train_imgs:
    import random
    random.seed(42)
    samples = random.sample(train_imgs, min(8, len(train_imgs)))
    
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    for ax, img_path in zip(axes.flat, samples):
        lbl_path = img_path.parent.parent / 'labels' / (img_path.stem + '.txt')
        show_annotated_image(img_path, lbl_path, ax)
    
    plt.suptitle('Sample Training Images with Annotations', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('No training images found. Run download_dataset.py first.')

## 5. Key Observations

After exploring the dataset, document your findings here:

1. **Class Balance**: Note any significant imbalance between classes
2. **Object Sizes**: Distribution of small vs. large objects affects model choice
3. **Annotation Quality**: Any issues with label consistency
4. **Scene Diversity**: Coverage of indoor/outdoor, lighting conditions
5. **Challenges**: Occlusion, crowded scenes, distant workers